In [ ]:
import scrapy
import pandas as pd
from scrapy.crawler import CrawlerProcess
from scrapy import Request
from scrapy.spiders import Spider
from datetime import datetime

In [1]:
import os
from mongodb_client import MongoDBClient
mongo_uri = os.getenv("MONGO_URI")
db_name = os.getenv("DB_NAME")
collection = os.getenv("COLLECTION_NAME")

In [2]:
client = MongoDBClient(mongo_uri, db_name, collection)

In [ ]:
class LaRazonSpider(Spider):
    name = "larazon"
    allowed_domains = ["www.la-razon.com"]
    start_urls = [
        f"https://www.la-razon.com/tags/feminicidio/page/{i}" for i in range(1, 88)
    ]
    def __init__(self):
        self.items = []
        self.mongo_client = client
        
    def date_formatter(self, date_str, date_format="%Y-%m-%d")
        try:
            date_concat = date_str[4] + date_str[5] + date_str[6]
            date_publish = datetime.strptime(date_concat, date_format)
            return date_publish
        except Exception as e:
            self.logger.error(f"Error al formatear fecha: {e}")
            return None
    
    def tittle_formatter(self, title):
        try:
            title = title[0].full_text
            title = title.replace("“", '"')
            title = title.replace("”", '"')
            return title
        except Exception as e:
            self.logger.error(f"Error al formatear título: {e}")
            return title
    
    def tag_formatter(self, tags):
        try:
            list_tags = [t.full_text.lower() for t in tags]
            return list_tags
        except Exception as e:
            self.logger.error(f"Error al formatear tags: {e}")
            return tags
    
    def section_formatter(self, url):
        try:
            url_split = url.split("/")
            section = url_split[3]
            return section
        except Exception as e:
            self.logger.error(f"Error al formatear sección: {e}")
            return url

    def body_formatter(self, body):
        try:
            new_body = [
                b.strip()
                .replace("\xa0", " ")
                .replace("\ufeff", " ")
                .replace("“", '"')
                .replace("”", '"')
                .replace("\u200b", " ")
                for b in new_body
            ]
            new_body = [b for b in new_body if b != " "]
            return new_body
        except Exception as e:
            self.logger.error(f"Error al formatear cuerpo: {e}")
            return body

    def start_requests(self):
        for url in self.start_urls:
            self.logger.info(f"Enviando request a: {url}")
            yield Request(url=url, callback=self.parse_response)
    
    def parse_response(self, response):
        self.logger.info(f"Recibida respuesta: {response.url}")
        try:
            pass
        except Exception as e:
            self.logger.error(f"Error al procesar la respuesta JSON: {e}")
            return